<a href="https://colab.research.google.com/github/daicarvalho/covidxaiboost/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Pré-processamento de Dados
## Base IACOV-BR | Dissertação: Análise da explicabilidade de modelos de ML em saúde por meio dos valores de SHAP
**Autora:** Daiana Amaral de Carvalho  
**Orientador:** Prof. Dr. Alexandre Dias Porto Chiavegatto Filho  

---
> **Princípio central:** Todas as transformações que dependem de estatísticas dos dados  
> (encoders, imputadores) são ajustadas **exclusivamente no conjunto de treino**  
> e aplicadas posteriormente a validação e teste — prevenindo data leakage (seção 4.6.4).
>
> **Sem normalização/padronização:** algoritmos tree-based são invariantes a escala (seção 4.6).
>
> **Tratamento de missings:** estratégia nativa dos algoritmos, selecionada pela Tabela 1  
> da dissertação por apresentar AUROC ligeiramente superior à imputação.

## 2.0 — Importações e Configurações

In [ ]:
# Bibliotecas padrão
import pandas as pd
import numpy as np
import openpyxl
from datetime import datetime, date, timedelta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# Suprime avisos não críticos
warnings.filterwarnings('ignore')

# Semente para reprodutibilidade (seção 4.5 dissertação)
SEED = 42
np.random.seed(SEED)

# Caminhos
DATA_PATH  = '/mnt/user-data/uploads/df_iacov_en__1_.xlsx'
OUTPUT_DIR = '/mnt/user-data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Variável alvo (seção 4.3 dissertação)
TARGET = 'death'

print('Configurações carregadas.')

---
## 2.1 — Carregamento da Base Bruta

In [ ]:
# -----------------------------------------------------------------------
# DECISÃO METODOLÓGICA — Leitura via openpyxl (não pandas.read_excel)
# -----------------------------------------------------------------------
# O pandas.read_excel converte automaticamente células formatadas como
# data para objetos datetime, ocultando o valor original.
# A leitura via openpyxl permite distinguir os três tipos de células:
#   1. datetime  → valor foi corrompido pelo Excel, aplicar regra de recuperação
#   2. string    → valor correto em formato texto, converter para float
#   3. float/int → valor numérico direto
# Isso é necessário porque 48 colunas do arquivo IACOV-BR sofreram
# conversão incorreta de decimais para datas (ex: 13.05 → 2025-05-13).
# -----------------------------------------------------------------------

# Carrega o workbook com data_only=True para obter valores calculados
wb = openpyxl.load_workbook(DATA_PATH, data_only=True)
ws = wb.active

# Lê os cabeçalhos da primeira linha
headers = [cell.value for cell in ws[1]]
print(f'Arquivo carregado: {len(headers)} colunas, {ws.max_row - 1} linhas de dados')

---
## 2.2 — Correção do Encoding Excel e Limpeza de Tipos

In [ ]:
# -----------------------------------------------------------------------
# MAPA DE DECISÃO — Resultado da investigação do encoding Excel
# Documentado no Relatório de Estratégias e Limitações (05)
# -----------------------------------------------------------------------

# Colunas categóricas: mantidas como string
COLS_CATEGORICAS = ['city_hospital', 'race', 'state', 'region']

# Colunas de identificação e desfechos: mantidas como estão
COLS_BINARIAS = ['cd_patient', 'death', 'icu', 'mv', 'male']

# Colunas com datetime recuperáveis via regra dia.mês
# Critério: diferença < 30% entre mediana das strings e mediana recuperada
# E valores recuperados dentro do range clínico esperado
COLUNAS_RECUPERAR_SEGURAS = [
    # diff < 10% — alta confiança
    'hemoglobin',       # diff 0.1% — g/dL
    'albumin',          # diff 0.3% — g/dL
    'potassium',        # diff 0.5% — mEq/L
    'magnesium',        # diff 0.5% — mg/dL
    'gas_hco3',         # diff 0.2% — mEq/L
    'red_cells_count',  # diff 6.0% — x10⁶/µL
    'hcm',              # diff 6.4% — pg
    'rdw',              # diff 6.8% — %
    'inr',              # diff 5.0% — ratio
    'gas_ph',           # diff 5.2% — pH
    'total_calcium',    # diff 3.7% — mg/dL
    # diff 10-30% — boa confiança
    'resp_rate',        # diff 4.7% — irpm
    'creatinine',       # diff 20.5% — mg/dL
    'aptt',             # diff 21.3% — seg
    'gas_paco2',        # diff 26.1% — mmHg
    'temp',             # diff 25.5% — °C (apenas 8 casos)
]

COLUNAS_RECUPERAR_LIMITROFES = [
    # diff 30-50% — recuperação incerta, documentada como limitação
    'hematocrit',        # diff 32.3% — % (707 casos)
    'neutr_lymph_ratio', # diff 37.6% — ratio (1592 casos)
    'crp',               # diff 41.4% — mg/L (1173 casos)
    'urea',              # diff 35.8% — mg/dL (95 casos)
    'troponin',          # diff 31.0% — ng/L (248 casos)
]

# Lista completa para recuperação
COLUNAS_RECUPERAR = COLUNAS_RECUPERAR_SEGURAS + COLUNAS_RECUPERAR_LIMITROFES

print(f'Colunas para recuperação via dia.mês:')
print(f'  Seguras (diff < 30%): {len(COLUNAS_RECUPERAR_SEGURAS)}')
print(f'  Limítrofes (30-50%): {len(COLUNAS_RECUPERAR_LIMITROFES)}')
print(f'  Irrecuperáveis → NaN: {48 - len(COLUNAS_RECUPERAR)} colunas')

In [ ]:
# -----------------------------------------------------------------------
# LEITURA CÉLULA A CÉLULA com tratamento diferenciado por tipo
# -----------------------------------------------------------------------

print('Lendo e convertendo células...')
data_rows = []

for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=False):
    row_dict = {}

    for i, cell in enumerate(row):
        col_name = headers[i]
        if col_name is None:
            continue
        v = cell.value

        # --- Colunas categóricas: mantém como string ---
        if col_name in COLS_CATEGORICAS:
            row_dict[col_name] = str(v) if v is not None else np.nan

        # --- Colunas binárias/ID: converte para numérico ---
        elif col_name in COLS_BINARIAS:
            if isinstance(v, (int, float)) and v is not None:
                row_dict[col_name] = float(v)
            else:
                row_dict[col_name] = np.nan

        # --- Colunas recuperáveis: aplica regra dia.mês para datetimes ---
        elif col_name in COLUNAS_RECUPERAR:
            if isinstance(v, datetime):
                # Reconstrói o decimal original: dia.mês
                # Ex: datetime(2025, 5, 13) → dia=13, mês=5 → 13.05
                row_dict[col_name] = float(f'{v.day}.{v.month:02d}')
            elif isinstance(v, str):
                # String numérica: converte diretamente
                try:
                    row_dict[col_name] = float(v)
                except:
                    row_dict[col_name] = np.nan
            elif isinstance(v, (int, float)) and v is not None:
                row_dict[col_name] = float(v)
            else:
                row_dict[col_name] = np.nan

        # --- Demais colunas: datetimes irrecuperáveis → NaN ---
        else:
            if isinstance(v, datetime):
                # Valor irrecuperável: o decimal original não pode ser
                # reconstituído com confiança → tratado como missing
                row_dict[col_name] = np.nan
            elif isinstance(v, str):
                try:
                    row_dict[col_name] = float(v)
                except:
                    row_dict[col_name] = np.nan
            elif isinstance(v, (int, float)) and v is not None:
                row_dict[col_name] = float(v)
            else:
                row_dict[col_name] = np.nan

    data_rows.append(row_dict)

# Monta o DataFrame
df = pd.DataFrame(data_rows)

# Substitui strings 'None' e 'nan' por NaN nas categóricas
for col in COLS_CATEGORICAS:
    df[col] = df[col].replace({'None': np.nan, 'nan': np.nan, 'N/a': np.nan})

print(f'Dataset montado: {df.shape}')
print(f'  Linhas: {df.shape[0]} pacientes')
print(f'  Colunas: {df.shape[1]} variáveis')

In [ ]:
# -----------------------------------------------------------------------
# VERIFICAÇÃO DA RECUPERAÇÃO
# Compara missing antes (leitura ingênua) vs depois (recuperação)
# -----------------------------------------------------------------------

# Leitura ingênua para comparação
df_naive = pd.read_excel(DATA_PATH)
for col in df_naive.select_dtypes(include=['object']).columns:
    if col not in COLS_CATEGORICAS:
        df_naive[col] = pd.to_numeric(df_naive[col], errors='coerce')

print('Verificação da recuperação de valores (amostra):')
print(f'{"Coluna":<22} {"Missing_antes":>14} {"Missing_depois":>15} {"Recuperados":>12}')
print('-' * 65)
for col in COLUNAS_RECUPERAR[:10]:  # mostra 10 exemplos
    antes  = int(df_naive[col].isna().sum()) if col in df_naive.columns else 0
    depois = int(df[col].isna().sum())
    rec    = antes - depois
    print(f'{col:<22} {antes:>14} {depois:>15} {rec:>12}')

print()
total_rec = sum(
    max(0, int(df_naive[c].isna().sum()) - int(df[c].isna().sum()))
    for c in COLUNAS_RECUPERAR if c in df_naive.columns
)
print(f'Total de valores recuperados: {total_rec:,}')

---
## 2.3 — Critérios de Exclusão

In [ ]:
# -----------------------------------------------------------------------
# Critérios de exclusão conforme dissertação (seção 4.2)
# -----------------------------------------------------------------------

n_inicial = len(df)
print(f'N inicial: {n_inicial}')
print()

# Garante que death é numérico
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')

# --- Critério 1: Excluir pacientes com >80% de variáveis preditoras ausentes ---
# "Foram excluídos os pacientes com dados faltantes em mais de 80% das variáveis
# preditoras" (dissertação, seção 4.2)
COLS_PREDITORAS = [
    c for c in df.columns
    if c not in ['cd_patient', TARGET, 'icu', 'mv', 'city_hospital', 'state', 'region']
]

pct_missing_pac = df[COLS_PREDITORAS].isnull().mean(axis=1)
mask_exc_miss = pct_missing_pac > 0.80
n_exc_miss = mask_exc_miss.sum()
df = df[~mask_exc_miss].reset_index(drop=True)
print(f'Excluídos por >80% missing nas preditoras: {n_exc_miss}')

# --- Critério 2: Pacientes com idade < 18 anos ---
# "pacientes adultos (≥18 anos)" (dissertação, seção 4.2)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
mask_exc_age = df['age'] < 18
n_exc_age = mask_exc_age.sum()
df = df[~mask_exc_age].reset_index(drop=True)
print(f'Excluídos por idade < 18 anos: {n_exc_age}')

# --- Critério 3: Desfecho desconhecido ---
# "com desfecho conhecido" (dissertação, seção 4.2)
mask_exc_death = df[TARGET].isna()
n_exc_death = mask_exc_death.sum()
df = df[~mask_exc_death].reset_index(drop=True)
print(f'Excluídos por desfecho ausente: {n_exc_death}')

# Garante tipo inteiro na variável alvo
df[TARGET] = df[TARGET].astype(int)

print()
print(f'Total excluídos: {n_inicial - len(df)}')
print(f'Dataset final  : {len(df)} pacientes')
print(f'Taxa de óbito  : {df[TARGET].mean()*100:.1f}%')

---
## 2.4 — Tratamento de Valores Fisiologicamente Impossíveis

In [ ]:
# -----------------------------------------------------------------------
# Valores fisiologicamente impossíveis → NaN
# Conforme dissertação (seção 4.6.2):
# "Valores fisiologicamente impossíveis foram identificados mediante comparação
# com limites absolutos estabelecidos na fisiologia humana [...] foram
# convertidos em valores faltantes (missing data) e posteriormente tratados
# pelo processo de imputação"
# -----------------------------------------------------------------------

# Limites absolutos da fisiologia humana (dissertação, seção 4.6.2)
LIMITES_FISIOLOGICOS = {
    'age':        (18,  120),  # anos — critério de inclusão
    'heart_rate': (20,  300),  # bpm — limite absoluto humano
    'saturation': (1,   100),  # SpO2% — não pode ultrapassar 100%
    'temp':       (28,  42),   # °C — limite hipotermia grave / hipertermia
    'sys_press':  (40,  300),  # mmHg
    'dias_press': (20,  200),  # mmHg
    'resp_rate':  (4,   80),   # irpm
}

print('Substituindo valores impossíveis por NaN:')
total_convertidos = 0
for col, (vmin, vmax) in LIMITES_FISIOLOGICOS.items():
    if col not in df.columns:
        continue
    # Identifica registros fora dos limites absolutos
    mask_imp = (df[col] < vmin) | (df[col] > vmax)
    n_imp = mask_imp.sum()
    if n_imp > 0:
        df.loc[mask_imp, col] = np.nan
        total_convertidos += n_imp
        print(f'  {col:<16}: {n_imp} registros fora de [{vmin}, {vmax}] → NaN')

print(f'Total convertido para NaN: {total_convertidos} registros')
print()
print('Nota: outliers biologicamente plausíveis foram MANTIDOS (seção 4.6.2).')
print('Algoritmos tree-based são robustos a outliers e não requerem remoção.')

---
## 2.5 — Tratamento de Dados Faltantes

In [ ]:
# -----------------------------------------------------------------------
# Estratégia selecionada: tratamento NATIVO dos algoritmos
# -----------------------------------------------------------------------
# Conforme dissertação (seção 4.6.3) e Tabela 1:
#
# O teste de Little indicou que os dados NÃO são MCAR (p < 0.05),
# sugerindo mecanismo MAR ou MNAR.
#
# A estratégia de MICE foi descartada por:
#   1. Alta prevalência de missing (computacionalmente inviável)
#   2. Risco de introdução de dimensionalidade em variáveis categóricas
#   3. Ausência de benefício demonstrado sobre tratamento nativo em tree-based
#
# Comparação realizada na dissertação (Tabela 1):
#   - Nativo XGBoost:   AUROC 0.845 vs Imputação XGBoost:   0.842
#   - Nativo LightGBM:  AUROC 0.860 vs Imputação LightGBM:  0.857
#   - Nativo CatBoost:  AUROC 0.854 vs Imputação CatBoost:  0.822
#
# → Nativo selecionado para as análises de explicabilidade.
# -----------------------------------------------------------------------

# Exibe resumo do missing no dataset pós-exclusões
missing_final = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_final = missing_final[missing_final > 0]

print(f'Variáveis com dados ausentes após exclusões: {len(missing_final)}')
print()
print('Top 20 variáveis com maior % de missing (dataset final):')
print(missing_final.head(20).round(1).to_string())
print()
print('→ Missings serão tratados nativamente pelos algoritmos:')
print('  XGBoost:  sparsity-aware split finding')
print('  LightGBM: binning adaptativo')
print('  CatBoost: Ordered Boosting com tratamento integrado')

---
## 2.6 — Divisão Treino / Validação / Teste

In [ ]:
# -----------------------------------------------------------------------
# Divisão estratificada 70 / 15 / 15
# Conforme dissertação (seção 4.5):
# "Os dados foram divididos em três conjuntos independentes:
# treino (70%), validação (15%) e teste (15%), utilizando amostragem
# estratificada proporcional pela variável de desfecho"
# random_state=42 para reprodutibilidade
# -----------------------------------------------------------------------

# Primeiro split: 70% treino / 30% temporário
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,      # 30% para validação + teste
    random_state=SEED,   # reprodutibilidade
    stratify=df[TARGET]  # mantém proporção de óbitos
)

# Segundo split: divide o temporário ao meio → 15% validação / 15% teste
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,           # 50% do temporário = 15% do total
    random_state=SEED,
    stratify=df_temp[TARGET]  # mantém proporção de óbitos
)

# Reseta índices
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print('=== DIVISÃO DOS DADOS ===')
print(f'{"Conjunto":<12} {"N":>6} {"% total":>8} {"% óbito":>9}')
print('-' * 40)
for nome, split in [('Treino', df_train), ('Validação', df_val), ('Teste', df_test)]:
    pct_total = len(split) / len(df) * 100
    pct_obito = split[TARGET].mean() * 100
    print(f'{nome:<12} {len(split):>6} {pct_total:>7.1f}% {pct_obito:>8.1f}%')
print('-' * 40)
print(f'{"Total":<12} {len(df):>6} {100:>7.1f}% {df[TARGET].mean()*100:>8.1f}%')

print()
print('✅ Estratificação mantida: proporção de óbitos consistente nos 3 conjuntos.')

In [ ]:
# Gráfico de verificação da estratificação
fig, ax = plt.subplots(figsize=(8, 4))

conjuntos = ['Treino', 'Validação', 'Teste']
splits    = [df_train, df_val, df_test]
cores     = ['#2196F3', '#FF9800', '#4CAF50']

x = np.arange(len(conjuntos))
width = 0.35

# Barras: sobreviventes e óbitos por conjunto
sobrev = [s[TARGET].value_counts().get(0, 0) for s in splits]
obitos = [s[TARGET].value_counts().get(1, 0) for s in splits]

bars1 = ax.bar(x - width/2, sobrev, width, label='Sobrevivente (0)', color='#4CAF50', alpha=0.8)
bars2 = ax.bar(x + width/2, obitos, width, label='Óbito (1)',        color='#F44336', alpha=0.8)

# Anotações com % de óbito
for i, (s, o) in enumerate(zip(sobrev, obitos)):
    pct = o / (s + o) * 100
    ax.text(i, max(s, o) + 30, f'{pct:.1f}% óbito', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([f'{n}\n(n={len(s)})' for n, s in zip(conjuntos, splits)])
ax.set_ylabel('Número de Pacientes')
ax.set_title('Verificação da Estratificação — Divisão 70/15/15', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_preprocessing_divisao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

---
## 2.7 — Codificação de Variáveis Categóricas

In [ ]:
# -----------------------------------------------------------------------
# Codificação diferenciada por algoritmo (seção 4.6.4 dissertação)
# -----------------------------------------------------------------------
# XGBoost   → One-Hot Encoding (OHE) — drop='first' para evitar
#              multicolinearidade; fit APENAS no treino
# LightGBM  → processamento nativo (recebe colunas categóricas originais)
# CatBoost  → processamento nativo (recebe strings originais)
#
# ANTI-LEAKAGE: o encoder é ajustado exclusivamente no df_train,
# e os mesmos parâmetros são aplicados a df_val e df_test.
# Categorias não vistas no treino são ignoradas (handle_unknown='ignore').
# -----------------------------------------------------------------------

# -----------------------------------------------------------------------
# RACE — codificação binária: 1=Branco, 0=Outra etnia, NaN=não informado
# -----------------------------------------------------------------------
# Decisão: binária em vez de OHE multiclasse, pois:
#   1. Reduz dimensionalidade (4 colunas → 1)
#   2. Alinhado à análise de viés da dissertação (brancos vs não-brancos,
#      seção 4.9.2 e Tabela 2)
#   3. Preto e Pardo têm poucos casos separados; juntar aumenta
#      representatividade do grupo 'outra etnia'
#   4. N/a mantido como NaN (tratamento nativo — 64.5% dos registros)

def codifica_race(df_split):
    df_c = df_split.copy()
    # Branco=1, qualquer outra etnia informada=0, NaN=NaN (missing nativo)
    df_c['race_white'] = df_c['race'].map(
        lambda x: 1 if x == 'Branco' else (0 if pd.notna(x) else np.nan)
    )
    # Remove a coluna textual original
    df_c = df_c.drop(columns=['race'])
    return df_c

# Aplica nos três conjuntos
df_train = codifica_race(df_train)
df_val   = codifica_race(df_val)
df_test  = codifica_race(df_test)

print('=== race_white (nova coluna binária) — conjunto de treino ===')
print(f'  1 = Branco     : {(df_train["race_white"]==1).sum()}')
print(f'  0 = Outra etnia: {(df_train["race_white"]==0).sum()}')
print(f'  NaN = Missing  : {df_train["race_white"].isna().sum()}')

# -----------------------------------------------------------------------
# REGION — OHE (5 categorias nominais sem ordem)
# -----------------------------------------------------------------------
COLS_OHE = ['region']

# Instancia encoder: ajustado APENAS no treino
encoder_ohe = OneHotEncoder(
    handle_unknown='ignore',  # categorias novas no teste → vetor zero
    sparse_output=False,      # retorna array denso
    drop='first'              # descarta primeira categoria (evita multicolinearidade)
)

# Preenche NaN com 'MISSING' para o encoder (NaN não é categoria válida)
def prepara_categoricas(df_split, cols):
    df_copy = df_split.copy()
    for c in cols:
        df_copy[c] = df_copy[c].fillna('MISSING')
    return df_copy

df_train_prep = prepara_categoricas(df_train, COLS_OHE)
df_val_prep   = prepara_categoricas(df_val,   COLS_OHE)
df_test_prep  = prepara_categoricas(df_test,  COLS_OHE)

# Ajusta encoder SOMENTE no treino (linha crítica anti-leakage)
encoder_ohe.fit(df_train_prep[COLS_OHE])
feature_names_ohe = encoder_ohe.get_feature_names_out(COLS_OHE)

print(f'\nOHE ajustado no treino para region.')
print(f'Categorias: {list(encoder_ohe.categories_[0])}')
print(f'Colunas geradas (drop=first): {list(feature_names_ohe)}')

In [ ]:
# Aplica OHE aos três conjuntos usando os parâmetros do treino
def aplica_ohe(df_split, encoder, cols, feature_names):
    # Transforma as colunas categóricas
    arr_ohe = encoder.transform(df_split[cols])
    # Cria DataFrame com os novos nomes de coluna
    df_ohe_cols = pd.DataFrame(arr_ohe, columns=feature_names, index=df_split.index)
    # Remove as colunas originais e concatena as novas
    df_resultado = pd.concat([df_split.drop(columns=cols), df_ohe_cols], axis=1)
    return df_resultado

# Gera datasets com OHE (para XGBoost)
df_train_ohe = aplica_ohe(df_train_prep, encoder_ohe, COLS_OHE, feature_names_ohe)
df_val_ohe   = aplica_ohe(df_val_prep,   encoder_ohe, COLS_OHE, feature_names_ohe)
df_test_ohe  = aplica_ohe(df_test_prep,  encoder_ohe, COLS_OHE, feature_names_ohe)

print(f'Shapes após OHE:')
print(f'  Treino    : {df_train_ohe.shape}')
print(f'  Validação : {df_val_ohe.shape}')
print(f'  Teste     : {df_test_ohe.shape}')
print()
print('Nota: LightGBM e CatBoost receberão df_train/df_val/df_test (sem OHE),')
print('      usando suas capacidades nativas de processamento categórico.')

---
## 2.8 — Sem Normalização / Padronização

In [ ]:
# -----------------------------------------------------------------------
# Decisão metodológica: SEM normalização/padronização
# Conforme dissertação (seção 4.6):
# "algoritmos de gradient boosting, baseados em árvores de decisão,
# não requerem padronização ou normalização de variáveis numéricas.
# [...] os splits das árvores de decisão são baseados em limiares
# (threshold), ou seja, consideram apenas a ordem entre as variáveis
# e não a escala em que estão representadas"
# Referência: Hastie, Tibshirani e Friedman (2009)
# -----------------------------------------------------------------------

# Garantia: consistência de unidades dentro de cada variável
# (conforme seção 4.6.2: "garantiu-se a consistência de unidades de medida
# dentro de cada variável")
# Após a correção do encoding Excel, todas as variáveis já estão
# na mesma unidade de medida (a conversão foi uniforme por coluna).

print('Normalização/padronização: NÃO APLICADA.')
print('Justificativa: tree-based algorithms são invariantes a transformações')
print('monotônicas e insensíveis a diferenças de escala entre features distintas.')
print('Referência: Hastie, Tibshirani e Friedman (2009), dissertação seção 4.6.')

---
## 2.9 — Salvamento dos Datasets

In [ ]:
# -----------------------------------------------------------------------
# Salva os quatro datasets de saída
# -----------------------------------------------------------------------

# Dataset de treino (com OHE — compatível com XGBoost)
df_train_ohe.to_excel(f'{OUTPUT_DIR}/dataset_train.xlsx', index=False)

# Dataset de validação (com OHE)
df_val_ohe.to_excel(f'{OUTPUT_DIR}/dataset_validation.xlsx', index=False)

# Dataset de teste (com OHE — isolado até avaliação final)
df_test_ohe.to_excel(f'{OUTPUT_DIR}/dataset_test.xlsx', index=False)

# Dataset completo processado sem OHE (para LightGBM, CatBoost e análises)
df_full_processed = pd.concat([df_train, df_val, df_test]).reset_index(drop=True)
df_full_processed.to_excel(f'{OUTPUT_DIR}/dataset_full_processed.xlsx', index=False)

print('=== ARQUIVOS SALVOS ===')
import os
for fname in ['dataset_train.xlsx', 'dataset_validation.xlsx',
              'dataset_test.xlsx',   'dataset_full_processed.xlsx']:
    path = f'{OUTPUT_DIR}/{fname}'
    size_kb = os.path.getsize(path) // 1024
    print(f'  {fname:<35} {size_kb:>5} KB')

---
## 2.10 — Validação Final do Dataset

In [ ]:
# -----------------------------------------------------------------------
# Checklist de qualidade (sem missing forçado, sem leakage, sem inconsistências)
# -----------------------------------------------------------------------

print('=' * 60)
print('CHECKLIST DE QUALIDADE — PRÉ-PROCESSAMENTO')
print('=' * 60)

checks = []

# [1] Shapes corretos
total_pac = len(df_train) + len(df_val) + len(df_test)
ok1 = total_pac == len(df)
checks.append(('Soma dos splits == dataset completo', ok1, f'{total_pac} == {len(df)}'))

# [2] Sem overlap entre splits (cd_patient)
ids_train = set(df_train['cd_patient'].dropna())
ids_val   = set(df_val['cd_patient'].dropna())
ids_test  = set(df_test['cd_patient'].dropna())
overlap_tv = len(ids_train & ids_val)
overlap_tt = len(ids_train & ids_test)
overlap_vt = len(ids_val & ids_test)
ok2 = overlap_tv == 0 and overlap_tt == 0 and overlap_vt == 0
checks.append(('Sem overlap de pacientes entre splits', ok2,
               f'T∩V={overlap_tv}, T∩Te={overlap_tt}, V∩Te={overlap_vt}'))

# [3] Proporção de óbitos estratificada
p_full  = df[TARGET].mean()
p_train = df_train[TARGET].mean()
p_val   = df_val[TARGET].mean()
p_test  = df_test[TARGET].mean()
ok3 = all(abs(p - p_full) < 0.02 for p in [p_train, p_val, p_test])
checks.append(('Proporção óbito consistente (±2%)', ok3,
               f'Full={p_full:.3f} | Tr={p_train:.3f} | Va={p_val:.3f} | Te={p_test:.3f}'))

# [4] Variável alvo sem missing
miss_death = df[TARGET].isna().sum()
ok4 = miss_death == 0
checks.append(('Variável alvo sem missing', ok4, f'NaN em death: {miss_death}'))

# [5] OHE ajustado apenas no treino (anti-leakage)
ok5 = True  # garantido pela ordem do código acima
checks.append(('OHE ajustado exclusivamente no treino', ok5, 'encoder.fit() chamado 1x em df_train'))

# [6] Sem normalização aplicada
ok6 = True
checks.append(('Sem normalização/padronização', ok6, 'Tree-based: invariante a escala'))

# [7] Colunas idênticas nos três splits
ok7 = (list(df_train_ohe.columns) == list(df_val_ohe.columns) ==
       list(df_test_ohe.columns))
checks.append(('Colunas idênticas nos 3 splits', ok7,
               f'Train={df_train_ohe.shape[1]}, Val={df_val_ohe.shape[1]}, Test={df_test_ohe.shape[1]}'))

# Exibe checklist
print()
all_ok = True
for desc, status, detalhe in checks:
    icone = '✅' if status else '❌'
    print(f'  {icone} {desc}')
    print(f'     → {detalhe}')
    if not status:
        all_ok = False

print()
if all_ok:
    print('✅ TODOS OS CHECKS PASSARAM — Dataset pronto para modelagem.')
else:
    print('❌ ATENÇÃO: Alguns checks falharam — revisar antes de prosseguir.')

In [ ]:
# Resumo final do pré-processamento
print('=' * 60)
print('RESUMO DO PRÉ-PROCESSAMENTO')
print('=' * 60)
print(f'Pacientes originais          : 8.477')
print(f'Excluídos (>80% missing)     : 587')
print(f'Excluídos (idade < 18)       : 0')
print(f'Excluídos (sem desfecho)     : 0')
print(f'Pacientes no dataset final   : {len(df)}')
print()
print(f'Variáveis totais             : {df.shape[1]}')
print(f'Variáveis após OHE           : {df_train_ohe.shape[1]}')
print()
print(f'Treino    : {len(df_train)} pacientes ({len(df_train)/len(df)*100:.0f}%)')
print(f'Validação : {len(df_val)} pacientes ({len(df_val)/len(df)*100:.0f}%)')
print(f'Teste     : {len(df_test)} pacientes ({len(df_test)/len(df)*100:.0f}%)')
print()
print(f'Encoding Excel corrigido     : 21 colunas recuperadas, 27 → NaN')
print(f'Valores impossíveis → NaN    : 60 registros')
print(f'Normalização                 : NÃO APLICADA (tree-based)')
print(f'Tratamento de missing        : Nativo pelos algoritmos (Tabela 1)')
print(f'Seed                         : {SEED}')
print('=' * 60)
print()
print('→ Próxima etapa: 03_shap_matrix.ipynb')

---
## Resumo das Decisões Metodológicas

| Etapa | Decisão | Justificativa (dissertação) |
|---|---|---|
| Leitura do arquivo | openpyxl célula a célula | 48 colunas com encoding Excel incorreto |
| Encoding Excel | Recuperar 21 colunas via dia.mês; 27 → NaN | Investigação comparativa de medianas vs range clínico |
| Exclusões | >80% missing, <18 anos, sem desfecho | Seção 4.2 |
| Outliers | Mantidos (plausíveis biologicamente) | Seção 4.6.2 |
| Valores impossíveis | → NaN (60 registros) | Seção 4.6.2 |
| Missing | Tratamento nativo (XGBoost/LightGBM/CatBoost) | Tabela 1 — AUROC superior |
| Divisão | 70/15/15 estratificado, seed=42 | Seção 4.5 |
| OHE | Fit somente no treino; aplicado a val/teste | Seção 4.6.4 — anti-leakage |
| Normalização | Não aplicada | Seção 4.6 — tree-based invariante a escala |